   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 23.0 MB/s eta 0:00:00


In [1]:
!pip install -q pymupdf flask pyngrok pandas openpyxl sentence-transformers pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 12.6 MB/s eta 0:00:00


In [ ]:



from pyngrok import ngrok
from flask import Flask, request, jsonify
import fitz, pandas as pd, base64, pathlib
from io import BytesIO
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone

ngrok.set_auth_token("3BwCj0yWJbUlFGb6Z7xNGD08PUV_mNAvf7qHxSryb8neJUhn")
pc = Pinecone(api_key="pcsk_68KBSE_5ZmTMXWJsVgmA6RPZCGEYf8cVCaujApX7eCdfYKfWXTBnCNKvnTRP6kQTaL8g4p ")
index = pc.Index("fallahtech")
app = Flask(__name__)

def pdf_to_markdown(pdf_bytes, filename):
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    pages_md = []
    for num_page, page in enumerate(doc, 1):
        raw = page.get_text('rawdict')
        chars = []
        for bloc in raw['blocks']:
            if bloc.get('type') != 0:
                continue
            for line in bloc['lines']:
                for span in line['spans']:
                    for char in span['chars']:
                        chars.append((round(char['origin'][1], 1), char['origin'][0], char['c']))
        if not chars:
            continue
        chars.sort(key=lambda t: (round(t[0]/3)*3, t[1]))
        lignes = {}
        for y0, x0, c in chars:
            y_key = round(y0 / 3) * 3
            if y_key not in lignes:
                lignes[y_key] = []
            lignes[y_key].append((x0, c))
        lignes_texte = []
        for y_key in sorted(lignes.keys()):
            pts = sorted(lignes[y_key], key=lambda t: t[0])
            colonnes = []
            col = pts[0][1]
            x_prev = pts[0][0]
            for x0, c in pts[1:]:
                gap = x0 - x_prev
                if gap > 40:
                    colonnes.append(col.strip())
                    col = c
                else:
                    col += c
                x_prev = x0
            colonnes.append(col.strip())
            colonnes = [c for c in colonnes if c]
            if len(colonnes) > 1:
                lignes_texte.append(' | '.join(colonnes))
            elif colonnes:
                lignes_texte.append(colonnes[0])
        pages_md.append(f'## {filename} — Page {num_page}\n\n' + '\n'.join(lignes_texte))
    doc.close()
    return '\n\n---\n\n'.join(pages_md)

def excel_to_markdown(excel_bytes, filename):
    xl = pd.read_excel(BytesIO(excel_bytes), sheet_name=None, header=None)
    out = []
    for feuille, df in xl.items():
        df = df.dropna(how='all').reset_index(drop=True)
        if df.empty:
            continue
        lignes_texte = []
        for _, row in df.iterrows():
            vals = [str(v).strip() for v in row if str(v).strip() not in ('', 'nan')]
            if not vals:
                lignes_texte.append('')
                continue
            if len(vals) == 1:
                lignes_texte.append(f'### {vals[0]}')
            else:
                lignes_texte.append(' | '.join(vals))
        out.append(f'## {filename} — Feuille : {feuille}\n\n' + '\n'.join(lignes_texte))
    return '\n\n'.join(out)

import re

def markdown_to_chunks(nom_fichier, texte_md):
    est_excel = nom_fichier.endswith('.xlsx')
    chunks = []

    sections = re.split(r'\n---\n|\n(?=## )', texte_md)

    for section in sections:
        section = section.strip()
        if not section or len(section) < 50:
            continue

        lignes = section.split('\n')
        titre = lignes[0].replace('##', '').strip()

        chunks.append({
            'text': section,
            'source': f'{nom_fichier} — {titre}',
            'fichier': nom_fichier,
            'type_doc': 'excel' if est_excel else 'pdf',
            'annees': '2025-2029' if est_excel else '2023-2025',
        })

    return chunks

@app.route('/parse', methods=['POST'])
def parse():
    try:
        filename = request.form.get('filename')
        file = request.files.get('file')
        file_bytes = file.read()
        print(f"Reçu : filename={filename}, taille={len(file_bytes)} bytes")

        if filename.endswith('.pdf'):
            text = pdf_to_markdown(file_bytes, filename)
            type_doc = 'pdf'
        else:
            text = excel_to_markdown(file_bytes, filename)
            type_doc = 'excel'

        # Chunking directement ici !
        chunks = markdown_to_chunks(filename, text)
        print(f"Chunks générés : {len(chunks)}")

        return jsonify({'chunks': chunks, 'source': filename, 'type_doc': type_doc})
    except Exception as e:
        print(f"ERREUR: {e}")
        return jsonify({'error': str(e)}), 500




public_url = ngrok.connect(5000)
print(f"✅ Endpoint n8n : {public_url}/parse")


app.run(port=5000)

✅ Endpoint n8n : NgrokTunnel: "https://ena-gallinaceous-sprawly.ngrok-free.dev" -> "http://localhost:5000"/parse
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:10:09] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:10:09] "POST /parse HTTP/1.1" 200 -


Reçu : filename==FallahTech_BusinessPlan_Complet.xlsx, taille=13167 bytes
Reçu : filename==4.1_Etude_Marche_Synthese.pdf, taille=276164 bytes
Reçu : filename==1.2_Contrat_Cooperative_Type.pdf, taille=252589 bytes
Chunks générés : 2
Chunks générés : 1
Reçu : filename== 0.0_Index_DataRoom.pdf, taille=251494 bytes
Reçu : filename==1.1_Statuts_FallahTech.pdf, taille=273502 bytes
Chunks générés : 1
Chunks générés : 1
Reçu : filename==Etats_Financiers_Historiques_NCT.pdf, taille=331930 bytes
Reçu : filename==2.1_Etats_Financiers_Historiques_NCT_2023_2025.pdf, taille=331930 bytes
Reçu : filename==3.1_Registre_Personnel.pdf, taille=271702 bytes
Chunks générés : 2
Chunks générés : 12
Chunks générés : 12


INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:10:09] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:10:09] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:10:09] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:10:09] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:10:09] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:10:09] "POST /parse HTTP/1.1" 200 -


Chunks générés : 6


...

Reçu : filename==1.2_Contrat_Cooperative_Type.pdf, taille=252589 bytes
Chunks générés : 1
Reçu : filename== 0.0_Index_DataRoom.pdf, taille=251494 bytes
Chunks générés : 1
Reçu : filename==1.1_Statuts_FallahTech.pdf, taille=273502 bytes
Reçu : filename==4.1_Etude_Marche_Synthese.pdf, taille=276164 bytes
Chunks générés : 1
Reçu : filename==2.1_Etats_Financiers_Historiques_NCT_2023_2025.pdf, taille=331930 bytes
Reçu : filename==Etats_Financiers_Historiques_NCT.pdf, taille=331930 bytes
Chunks générés : 2
Reçu : filename==3.1_Registre_Personnel.pdf, taille=271702 bytes
Chunks générés : 2
Chunks générés : 12
Chunks générés : 12


INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:16:22] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:16:22] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:16:22] "POST /parse HTTP/1.1" 200 -


Reçu : filename==FallahTech_BusinessPlan_Complet.xlsx, taille=13167 bytes
Chunks générés : 6
Reçu : filename==1.2_Contrat_Cooperative_Type.pdf, taille=252589 bytes
Chunks générés : 1
Reçu : filename== 0.0_Index_DataRoom.pdf, taille=251494 bytes
Reçu : filename==4.1_Etude_Marche_Synthese.pdf, taille=276164 bytes
Chunks générés : 1
Chunks générés : 2
Reçu : filename==Etats_Financiers_Historiques_NCT.pdf, taille=331930 bytes
Reçu : filename==2.1_Etats_Financiers_Historiques_NCT_2023_2025.pdf, taille=331930 bytes
Reçu : filename==1.1_Statuts_FallahTech.pdf, taille=273502 bytes
Reçu : filename==3.1_Registre_Personnel.pdf, taille=271702 bytes
Chunks générés : 2


INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:16:22] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:16:22] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:16:22] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:16:22] "POST /parse HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2026 00:16:22] "POST /parse HTTP/1.1" 200 -


Chunks générés : 1
Chunks générés : 12
Chunks générés : 12
